This code is to extract text from PDF or scanned PDF documents using OCR (Optical Character Recognition).

## Run OCR:

* For general printed text: use Tesseract via ```pytesseract.image_to_string.```
* For PDF files: simply use ```ocrmypdf.ocr()``` to automate the above (it will do steps 1–3 internally
ocrmypdf.readthedocs.io
).
* For historical or difficult fonts: consider using Kraken or Calamari (after segmenting pages into lines).

After extraction, you may need to clean up common OCR errors (spell-check, regex fixes for old characters). Some engines (Calamari) output confidence scores or allow voting to improve output.

* ```dpi=150``` → faster, lower quality, may be fine for very clear PDFs.
* ```dpi=200``` → balance of speed and readability, often enough for simple text.
* ```dpi=300``` → standard OCR quality (most recommended).
* ```dpi=400+``` → sometimes improves OCR for poor scans, but increases memory/time usage.

* It took 3.4s to run this code for page no 73 to 77 (5 pages) with dpi = 150
* It took 7.0s to run this code for page no 73 to 77 (5 pages) with dpi = 200
* It took 11.4s to run this code for page no 73 to 77 (5 pages) with dpi = 250
* It took 13.9s to run this code for page no 73 to 77 (5 pages) with dpi = 300

22 Minute to extract Vol 1 PDF of The History and Culture of the India by RC Majmudar

### The History And Culture of the India

In [ ]:

import pandas as pd
import re
from pdf2image import convert_from_path
import pytesseract
import time

start = time.time()          # start time

pdf_path = "/Downloads/Project_Book/The History and Culture of the India/The History and Culture of Indian People Volume 10.pdf"

start_page = 23
end_page = 646


start1 = time.time()

documents = []
current_chapter = None
current_title = None

for i in range(start_page, end_page + 1):
    # Convert only the current page to an image
    # convert_from_path returns a list, even for a single page
    page_images = convert_from_path(
        pdf_path,
        dpi=150,
        first_page=i,
        last_page=i
    )

    if not page_images:
        print(f"Could not convert page {i}. Skipping.")
        continue

    image = page_images[0] # Get the single image for the current page

    text = pytesseract.image_to_string(image)
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    # Check first few lines for a chapter header
    # Retain current_chapter and current_title across pages unless a new one is found
    found_new_chapter_on_page = False
    for idx, line in enumerate(lines[:5]):  # only look at first 5 lines
        match = re.search(r"(?:CHAPTER)\s+([0-9IVXLC]+)", line)
        if match:
            current_chapter = f"Chapter {match.group(1)}"
            found_new_chapter_on_page = True

            # Prepare to collect title lines
            title_parts = []
            next_idx = idx + 1

            # Check the next line for valid title (skip if it starts with "I." or "1." or "A.")
            if next_idx < len(lines):
                next_line = lines[next_idx].strip()
                if not next_line.startswith(("I.", "1.", "A.")):
                    title_parts.append(next_line)
                    # Also check one more line for continuation
                    next_idx += 1
                    if next_idx < len(lines):
                        next_line = lines[next_idx].strip()
                        words = next_line.split()
                        if words and words[0].isupper() and not next_line.startswith(("I.", "1.", "A.")):
                            title_parts.append(next_line)

            # Combine parts into one title string
            if title_parts:
                current_title = " ".join(title_parts).title()
            else:
                # If a chapter is found but no immediate title, reset title to None for this page
                current_title = None
            break # Found a chapter, stop checking first few lines

    doc = {
        "chapter": current_chapter,
        "title": current_title,
        "page": i,
        "content": text
    }
    documents.append(doc)
    docs = pd.DataFrame(documents)
    docs.to_csv("10.British_Paramountcy_And_Indian_Renaissance_2.csv", index=False, encoding="utf-8-sig")
    print(f"Text from page {i}")



end1 = time.time()            # end time
elapsed1 = end1 - start1
print("Took", elapsed1/60, "minutes to Extract text.")

### Special Case: Andre Wink AL Hind Vol 2 : 2 page in 1 book

In [ ]:
import pandas as pd
import re
from pdf2image import convert_from_path
import pytesseract
import time
from PIL import Image

start = time.time()

pdf_path = "/Downloads/Project_Book/Indian History/Nationalist/New Upload/Andre_Wink_Al_Hind_Vol2.pdf"

start_page = 15
end_page = 377

documents = []
current_chapter = None
current_title = None

logical_page_number = 1  # actual book page counter

def extract_chapter_title(text, current_chapter, current_title):
    lines = [line.strip() for line in text.splitlines() if line.strip()]

    for idx, line in enumerate(lines[:5]):
        match = re.search(r"(?:CHAPTER)\s+([0-9IVXLC]+)", line)
        if match:
            current_chapter = f"Chapter {match.group(1)}"

            title_parts = []
            next_idx = idx + 1

            if next_idx < len(lines):
                next_line = lines[next_idx]
                if not next_line.startswith(("I.", "1.", "A.")):
                    title_parts.append(next_line)
                    next_idx += 1
                    if next_idx < len(lines):
                        cont_line = lines[next_idx]
                        words = cont_line.split()
                        if words and words[0].isupper():
                            title_parts.append(cont_line)

            current_title = " ".join(title_parts).title() if title_parts else None
            break

    return current_chapter, current_title


for i in range(start_page, end_page + 1):

    images = convert_from_path(
        pdf_path,
        dpi=150,
        first_page=i,
        last_page=i
    )

    if not images:
        print(f"Skipping page {i}")
        continue

    img = images[0]
    width, height = img.size

    # Split into left and right halves
    left_img = img.crop((0, 0, width // 2, height))
    right_img = img.crop((width // 2, 0, width, height))

    for page_img in [left_img, right_img]:
        text = pytesseract.image_to_string(page_img)

        if not text.strip():
            continue

        current_chapter, current_title = extract_chapter_title(
            text, current_chapter, current_title
        )

        documents.append({
            "chapter": current_chapter,
            "title": current_title,
            "pdf_page": i,
            "book_page": logical_page_number,
            "content": text
        })

        logical_page_number += 1


    pd.DataFrame(documents).to_csv(
        "Andre_Wink_Al_Hind_Vol2.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print(f"Processed PDF page {i}")

end = time.time()
print("Took", (end - start) / 60, "minutes")
